# Use GPT 4.1 Mini to generate semantic queries using keywords

# Import Libraries

In [1]:
import ast
import random 
random.seed(1234)

import numpy as np
import pandas as pd

from utils.gpt_query_utils import predict

# Manually extract the non-name keywords to constructing semantic query

In [2]:
keywords_df = pd.read_csv("keywords_cleaned.csv")
keywords_df = keywords_df[~keywords_df["Keywords"].isnull()]
len(keywords_df)

113

In [3]:
def clean_and_parse(val):
    if isinstance(val, str):
        # Replace curly quotes with standard quotes
        val = val.replace("“", '"').replace("”", '"').replace("‘", "'").replace("’", "'")
        try:
            return ast.literal_eval(val)
        except Exception:
            return []  # or: return val if you want to keep the original
    return val

keywords_df["keywords"] = keywords_df["Keywords"].apply(clean_and_parse)
keywords = keywords_df["keywords"].drop_duplicates().to_list()
len(keywords)

112

In [4]:
keywords[:2]

[['HIPPA', 'insurance'], ['tax preparation services']]

## Randomly select keywords to be included in the query for hybrid search

In [5]:
included_words = []
for group in keywords:
    sample = random.sample(group, k=random.randint(0, len(group)))
    included_words.append(sample)

In [6]:
included_words[:2]

[['HIPPA'], []]

In [7]:
assert len(keywords) == len(included_words)

In [8]:
keywords_dict = {"keywords": keywords,
                 "included_words": included_words}

# Use GPT to construct queries

In [10]:
results = predict(keywords_dict)
results

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"

[{'model': 'gpt-4.1-mini-2025-04-14',
  'input_tokens': 349,
  'output_tokens': 34,
  'raw_results': '{"query": "How does \'HIPPA\' protect patient information when dealing with insurance companies, and what legal obligations do insurers have to comply with these privacy rules?"}',
  'parsed_results': {'query': "How does 'HIPPA' protect patient information when dealing with insurance companies, and what legal obligations do insurers have to comply with these privacy rules?"}},
 {'model': 'gpt-4.1-mini-2025-04-14',
  'input_tokens': 344,
  'output_tokens': 25,
  'raw_results': '{"query": "What are the legal requirements and consumer protections applicable to companies offering tax preparation services in the United States?"}',
  'parsed_results': {'query': 'What are the legal requirements and consumer protections applicable to companies offering tax preparation services in the United States?'}},
 {'model': 'gpt-4.1-mini-2025-04-14',
  'input_tokens': 344,
  'output_tokens': 35,
  'raw_r

# Combine the keywords and queries 

In [11]:
input_df = pd.DataFrame(keywords_dict)

model_df = pd.DataFrame(results)

model_df["query"] = model_df["parsed_results"].apply(lambda x: x.get("query") if isinstance(x, dict) else None)

combined_df = pd.concat([input_df, model_df], axis=1)
combined_df.head()

,keywords,included_words,model,input_tokens,output_tokens,raw_results,parsed_results,query
0,"[HIPPA, insurance]",[HIPPA],gpt-4.1-mini-2025-04-14,349,34,"{""query"": ""How does 'HIPPA' protect patient in...",{'query': 'How does 'HIPPA' protect patient in...,How does 'HIPPA' protect patient information w...
1,[tax preparation services],[],gpt-4.1-mini-2025-04-14,344,25,"{""query"": ""What are the legal requirements and...",{'query': 'What are the legal requirements and...,What are the legal requirements and consumer p...
2,[personal tax return],[],gpt-4.1-mini-2025-04-14,344,35,"{""query"": ""What are the legal requirements and...",{'query': 'What are the legal requirements and...,What are the legal requirements and deadlines ...
3,[warranty of habitability],[],gpt-4.1-mini-2025-04-14,346,29,"{""query"": ""What does the 'warranty of habitabi...",{'query': 'What does the 'warranty of habitabi...,What does the 'warranty of habitability' requi...
4,[illegal contract],[],gpt-4.1-mini-2025-04-14,343,30,"{""query"": ""What constitutes an illegal contrac...",{'query': 'What constitutes an illegal contrac...,What constitutes an illegal contract under U.S...


In [12]:
# cost of generating the queries using gpt4.1-mini
combined_df["input_tokens"].sum()/1000000*0.4+combined_df["output_tokens"].sum()/1000000*1.6

np.float64(0.0227068)

## Clean the df a bit

In [13]:
combined_df = combined_df[['keywords', 'included_words', 'query']]

In [14]:
def clean_query(row):
    if isinstance(row["included_words"], list) and len(row["included_words"]) == 0:
        # Step 1: remove single quotes entirely
        cleaned = row["query"].replace("'", "")
        # Step 2: replace all remaining single quotes with double quotes (if any were missed)
    else:
        cleaned = row["query"].replace("'", '"')
    return cleaned

# Apply the function row-wise
combined_df["query"] = combined_df.apply(clean_query, axis=1)
combined_df.head()

,keywords,included_words,query
0,"[HIPPA, insurance]",[HIPPA],"How does ""HIPPA"" protect patient information w..."
1,[tax preparation services],[],What are the legal requirements and consumer p...
2,[personal tax return],[],What are the legal requirements and deadlines ...
3,[warranty of habitability],[],What does the warranty of habitability require...
4,[illegal contract],[],What constitutes an illegal contract under U.S...


In [15]:
len(combined_df)

112

In [16]:
len(combined_df[combined_df["included_words"].apply(lambda x: isinstance(x, list) and len(x) == 0)])

53

# Save the df to be combined with other query params (filters etc)

In [21]:
combined_df.to_csv("gpt_queries.csv", index=False)